# 01 - PPO Expert Training (M1, RQ1)

**Group members:** TODO

Train a PPO expert on `Walker2d-v4` and `Ant-v4` to the spec thresholds (Walker2d > 3000, Ant > 4000). Deliverable: TensorBoard return curves, best checkpoint, hyperparameter discussion.

In [ ]:
# Make src importable whether run from notebooks/ or project root
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
# On Colab: mount Drive and set PROJECT_DATA_ROOT before importing src
from src import config, seeding, envs, collect, eval, plotting
seeding.set_seed(0)
ENV_ID = 'Walker2d-v4'  # switch to 'Ant-v4' for the second environment
print('device', config.device(), '| env', ENV_ID)

## Training configuration

Defaults follow the spec's Table 1. For the hyperparameter pilots, run short budgets (e.g. 500k steps) over gamma, gae_lambda, lr schedule, n_steps, ent_coef; lock the best, then run the full budget. Document each choice and why (M1 requirement).

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

MODEL_DIR = config.MODELS_DIR / f'ppo_expert_{ENV_ID}'
LOG_DIR   = config.LOGS_DIR / f'ppo_expert_{ENV_ID}'
N_ENVS = 4
TOTAL_TIMESTEPS = 2_000_000  # Walker2d; raise for Ant. Use ~50_000 for a pilot.

vec_env  = envs.make_vec(ENV_ID, n_envs=N_ENVS, seed=0)
eval_env = envs.make_vec(ENV_ID, n_envs=1, seed=99)

# NOTE: EvalCallback.eval_freq counts rollout steps, not env steps, so
# divide by N_ENVS to evaluate every ~20k environment steps.
eval_callback = EvalCallback(
    eval_env, best_model_save_path=str(MODEL_DIR), log_path=str(LOG_DIR),
    eval_freq=max(20_000 // N_ENVS, 1), n_eval_episodes=10,
    deterministic=True, render=False)
checkpoint_callback = CheckpointCallback(
    save_freq=max(100_000 // N_ENVS, 1), save_path=str(MODEL_DIR),
    name_prefix='ppo_checkpoint')

In [ ]:
model = PPO(
    'MlpPolicy', vec_env, n_steps=2048, batch_size=64, n_epochs=10,
    learning_rate=3e-4, clip_range=0.2, gamma=0.99, gae_lambda=0.95,
    ent_coef=0.0, vf_coef=0.5, max_grad_norm=0.5, verbose=1,
    tensorboard_log=str(LOG_DIR), seed=0,
    policy_kwargs=dict(net_arch=config.NET_ARCH))

model.learn(total_timesteps=TOTAL_TIMESTEPS,
            callback=[eval_callback, checkpoint_callback],
            tb_log_name='PPO_expert', progress_bar=True)
model.save(MODEL_DIR / 'ppo_expert_final')
print('saved best model to', MODEL_DIR)

## Evaluate the best checkpoint

Load `best_model` (saved by EvalCallback, not the final model) and confirm the threshold before Phase 2. Monitor live with `tensorboard --logdir logs/` and watch `rollout/ep_rew_mean`.

In [ ]:
mean, std = eval.evaluate(PPO.load(MODEL_DIR / 'best_model'), ENV_ID)
print(f'expert mean reward: {mean:.1f} +/- {std:.1f}')
print('threshold', config.RETURN_TARGETS[ENV_ID],
      '->', 'PASS' if mean > config.RETURN_TARGETS[ENV_ID] else 'NOT YET')